# Using Lacuna with Docker

This guide demonstrates how to run Lacuna analyses using the pre-built [Docker](https://docs.docker.com/) image, without installing Lacuna or its dependencies locally. We run a structural network mapping (SNM) analysis using the HCP1065 tractogram as a complete example.

## Prerequisites

You need Docker installed on your system. See [Get Docker](https://docs.docker.com/get-docker/) for installation instructions.

The Lacuna Docker image includes all dependencies (Python, MRtrix3, TemplateFlow templates) so you do not need to install anything else.

## Setup

Pull the Lacuna Docker image.

In [10]:
!docker pull ghcr.io/m-petersen/lacuna:edge
!docker tag ghcr.io/m-petersen/lacuna:edge lacuna

edge: Pulling from m-petersen/lacuna

6da507ff: Pulling fs layer 
007a5b45: Pulling fs layer 
92e2619d: Pulling fs layer 
aba74b8c: Pulling fs layer 
d371ca27: Pulling fs layer 
3ab417f2: Pulling fs layer 
324443e6: Pulling fs layer 
84eca1ef: Pulling fs layer 
d371ca27: Waiting fs layer 
324443e6: Waiting fs layer 
97e5e950: Waiting fs layer 
Digest: sha256:86fd97198a432d28f2400d9a00784e7c642156a4fa38001498e5393571368b29[2K
Status: Downloaded newer image for ghcr.io/m-petersen/lacuna:edge
ghcr.io/m-petersen/lacuna:edge


Verify the image works.

In [11]:
!docker run --rm lacuna --help

usage: lacuna [-h] [--version] <command> ...

Lacuna v0.0.1.dev385

options:
  -h, --help  show this help message and exit
  --version   show program's version number and exit

commands:
  Use 'lacuna <command> --help' for more information.

  <command>
    fetch     Download and setup connectomes
    run       Run lesion network mapping analyses
    collect   Aggregate parcelstats across subjects
    info      Display available resources (atlases, connectomes)
    bidsify   Convert NIfTI files to BIDS format
    tutorial  Setup tutorial data for learning Lacuna

Commands:
  bidsify   Convert NIfTI files to BIDS format
  fetch     Download and setup connectomes for analysis
  run       Run lesion analyses
  collect   Aggregate results across subjects
  info      Display available resources (atlases, connectomes)
  tutorial  Setup tutorial data for learning Lacuna

Examples:
  lacuna tutorial ./my_tutorial
  lacuna fetch gsp1000 --api-key \$DATAVERSE_API_KEY
  lacuna run rd /bids /outpu

## How volume mounts work

Docker containers are isolated from the host filesystem. To give the container access to your data, you mount host directories into the container using `-v`:

```
-v /host/path:/container/path:ro   # read-only
-v /host/path:/container/path      # read-write
```

| Purpose | Host path | Container path | Mode |
|---------|-----------|----------------|------|
| BIDS input data | `/path/to/bids` | `/bids` | read-only (`:ro`) |
| Output results | `/path/to/output` | `/output` | read-write |
| Connectomes | `/path/to/connectomes` | `/connectomes` | read-only (`:ro`) |

The container entrypoint is the `lacuna` command, so you pass subcommands directly after the image name.

## Prepare tutorial data

We use Lacuna inside the container to create a tutorial dataset. The `--rm` flag removes the container after it exits.

In [12]:
!mkdir -p /tmp/docker_tutorial
!docker run --rm -u $(id -u):$(id -g)\
    -v /tmp/docker_tutorial:/data \
    lacuna \
    tutorial /data/bids --force


Setting up tutorial data at: /data/bids
✓ Tutorial data copied to: /data/bids

The tutorial dataset includes:
  - 3 synthetic subjects (sub-01, sub-02, sub-03)
  - Binary lesion masks in MNI152NLin6Asym space
  - BIDS-compliant structure


In [13]:
!ls /tmp/docker_tutorial/bids/

dataset_description.json  participants.tsv  sub-01  sub-03
__init__.py		  __pycache__	    sub-02


Fetch the connectome.

In [14]:
!docker run --rm -u $(id -u):$(id -g)\
    -v /tmp/docker_tutorial:/data \
    lacuna \
    fetch hcp1065 --output-dir /data/connectomes

Fetching HCP1065 structural tractogram...
  Output: /data/connectomes
  Keep original: True


✓ HCP1065 fetch complete!
  Files: 1
  Duration: 0.0s
  Output: /data/connectomes


In [15]:
!ls /tmp/docker_tutorial/connectomes/

hcp1065_avg_tracts_trk.zip  hcp1065.tck  hcp1065_tracts


## Run structural network mapping

In [16]:
!mkdir -p /tmp/docker_tutorial/output
!docker run --rm -u $(id -u):$(id -g)\
    -v /tmp/docker_tutorial/bids:/bids:ro \
    -v /tmp/docker_tutorial/connectomes:/connectomes:ro \
    -v /tmp/docker_tutorial/output:/output \
    lacuna \
    run snm /bids /output \
    --connectome-path /connectomes/hcp1065.tck \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym

Inspect the output. The results are written to the host filesystem via the volume mount.

In [17]:
!ls /tmp/docker_tutorial/output/sub-01/ses-01/anat/

sub-01_ses-01_desc-provenance.json
sub-01_ses-01_label-acuteinfarct_summarystatistics_stats.json
sub-01_ses-01_space-MNI152NLin2009cAsym_label-acuteinfarct_mask.nii.gz
sub-01_ses-01_space-MNI152NLin6Asym_label-acuteinfarct_desc-snm_disconnectionmap.json
sub-01_ses-01_space-MNI152NLin6Asym_label-acuteinfarct_desc-snm_disconnectionmap.nii.gz
